# Cylinder Monitor — Signal Analysis Workbench
Load a WAV recording, visualize both channels, tune detection parameters, and see what the algorithm finds.

**Workflow:**
1. Drop a WAV file into `test_data/`
2. Set the filename in the Config cell
3. Run all cells top to bottom
4. Tune parameters in the Tuning cell and re-run from there

In [1]:
import numpy as np
import scipy.io.wavfile as wav
import scipy.signal as signal
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from pathlib import Path
from pydub import AudioSegment

pio.renderers.default = 'iframe'

DARK = dict(
    paper_bgcolor='#1a1a1a', plot_bgcolor='#111111',
    font_color='#cccccc',
    xaxis=dict(gridcolor='#2a2a2a', zerolinecolor='#444'),
    yaxis=dict(gridcolor='#2a2a2a', zerolinecolor='#444'),
)

print('Packages loaded OK')

Packages loaded OK


## Config — set your filename and parameters here

In [2]:
# ── File ──────────────────────────────────────────────────────────────────────
WAV_FILE = '../test_data/Test123.m4a'   # <-- change this to your file

# ── Detection parameters (mirror the app Settings tab) ───────────────────────
STROKE_IN        = 1.0    # stroke length in inches
IMPACT_MULT      = 10     # threshold multiplier for T_end detection
BREAKAWAY_MULT   = 3      # threshold multiplier for T_start search
DEBOUNCE_MS      = 50     # minimum ms between two spikes
MIN_LOOKBACK_MS  = 15     # minimum ms before T_end to search for T_start
MAX_LOOKBACK_MS  = 100    # maximum ms before T_end to search for T_start
HF_BIN_LOW       = 10     # FFT bin low (~ 1 kHz at 48kHz/480 samples)
HF_BIN_HIGH      = 100    # FFT bin high (~10 kHz)
HF_FLOOR         = 0.01   # minimum HF energy to pass the FFT gate
BASELINE_PCT     = 50     # percentile of RMS history used as baseline
CHUNK_MS         = 10     # RMS chunk size in ms (must match app)

# ── Signal combination method ─────────────────────────────────────────────────
# 'additive'       |Ch0| + |Ch1|  — use for laptop/phone mic
# 'multiplicative' |Ch0| x |Ch1|  — use for wireless mics mounted on cylinder
METHOD = 'additive'

## Load audio file (WAV or M4A)

In [3]:
path = Path(WAV_FILE)
if not path.exists():
    raise FileNotFoundError(f"File not found: {path.resolve()}\nDrop a file into test_data/ and update WAV_FILE above.")

suffix = path.suffix.lower()

if suffix in ('.m4a', '.mp4', '.aac', '.ogg', '.flac', '.mp3'):
    audio = AudioSegment.from_file(path)
    sr    = audio.frame_rate
    samples = np.array(audio.get_array_of_samples(), dtype=np.float32)
    # pydub interleaves channels: [L, R, L, R, ...]
    if audio.channels == 2:
        samples = samples.reshape(-1, 2)
    else:
        samples = np.stack([samples, samples], axis=1)
        print('Mono file — duplicated to stereo')
    # Normalise to [-1, 1]
    max_val = float(2 ** (8 * audio.sample_width - 1))
    data = samples / max_val
else:
    sr, data = wav.read(path)
    if data.dtype == np.int16:
        data = data.astype(np.float32) / 32768.0
    elif data.dtype == np.int32:
        data = data.astype(np.float32) / 2147483648.0
    else:
        data = data.astype(np.float32)
    if data.ndim == 1:
        data = np.stack([data, data], axis=1)
        print('Mono file — duplicated to stereo')

ch0 = data[:, 0]
ch1 = data[:, 1]
duration_s = len(ch0) / sr
t = np.linspace(0, duration_s, len(ch0))

print(f'File:      {path.name}')
print(f'Format:    {suffix}')
print(f'Rate:      {sr} Hz')
print(f'Duration:  {duration_s*1000:.1f} ms  ({duration_s:.2f} s)')
print(f'Samples:   {len(ch0):,}')
print(f'Ch0 peak:  {np.max(np.abs(ch0)):.4f}')
print(f'Ch1 peak:  {np.max(np.abs(ch1)):.4f}')

File:      Test123.m4a
Format:    .m4a
Rate:      48000 Hz
Duration:  3648.0 ms  (3.65 s)
Samples:   175,104
Ch0 peak:  0.1486
Ch1 peak:  0.0267


## Raw Waveforms — both channels

In [4]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Ch0 (Left)', 'Ch1 (Right)'))
fig.add_trace(go.Scatter(x=t*1000, y=ch0, mode='lines', line=dict(color='#4fc3f7', width=0.8), name='Ch0'), row=1, col=1)
fig.add_trace(go.Scatter(x=t*1000, y=ch1, mode='lines', line=dict(color='#81d4fa', width=0.8), name='Ch1'), row=2, col=1)
fig.update_layout(title='Raw Waveforms', height=500, **DARK)
fig.update_xaxes(title_text='Time (ms)', row=2, col=1, gridcolor='#2a2a2a')
fig.update_yaxes(gridcolor='#2a2a2a')
fig.show()

## Combined Signal — additive vs multiplicative

In [5]:
combined_add  = np.abs(ch0) + np.abs(ch1)
combined_mult = np.abs(ch0) * np.abs(ch1)

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Additive |Ch0|+|Ch1|', 'Multiplicative |Ch0|×|Ch1|'))
fig.add_trace(go.Scatter(x=t*1000, y=combined_add,  mode='lines', line=dict(color='#66bb6a', width=0.8), name='Additive'),       row=1, col=1)
fig.add_trace(go.Scatter(x=t*1000, y=combined_mult, mode='lines', line=dict(color='#ffa726', width=0.8), name='Multiplicative'), row=2, col=1)
fig.update_layout(title='Combined Signal', height=500, **DARK)
fig.update_xaxes(title_text='Time (ms)', row=2, col=1, gridcolor='#2a2a2a')
fig.update_yaxes(gridcolor='#2a2a2a')
fig.show()

## Rolling RMS — what the app sees

In [6]:
combined = combined_add if METHOD == 'additive' else combined_mult
chunk_samples = int(CHUNK_MS / 1000 * sr)

n_chunks = len(combined) // chunk_samples
rms_vals  = np.array([
    np.sqrt(np.mean(combined[i*chunk_samples:(i+1)*chunk_samples]**2))
    for i in range(n_chunks)
])
t_chunks = np.array([(i + 0.5) * CHUNK_MS for i in range(n_chunks)])

baseline  = float(np.percentile(rms_vals, BASELINE_PCT))
threshold = baseline * IMPACT_MULT
bkwy_thr  = baseline * BREAKAWAY_MULT

print(f'Method:    {METHOD}')
print(f'Chunks:    {n_chunks}  ({CHUNK_MS}ms each)')
print(f'Baseline:  {baseline:.6f}  ({BASELINE_PCT}th percentile)')
print(f'Threshold: {threshold:.6f}  ({IMPACT_MULT}× baseline)')
print(f'Breakaway: {bkwy_thr:.6f}  ({BREAKAWAY_MULT}× baseline)')

fig = go.Figure()
fig.add_trace(go.Scatter(x=t_chunks, y=rms_vals, mode='lines', line=dict(color='#4fc3f7', width=1), name='RMS'))
fig.add_hline(y=threshold, line=dict(color='#ef5350', width=1.5, dash='dash'), annotation_text=f'Impact threshold ({IMPACT_MULT}×)')
fig.add_hline(y=bkwy_thr,  line=dict(color='#ffa726', width=1,   dash='dot'),  annotation_text=f'Breakaway ({BREAKAWAY_MULT}×)')
fig.add_hline(y=baseline,  line=dict(color='#555555', width=1),                annotation_text=f'Baseline ({BASELINE_PCT}th pct)')
fig.update_layout(title='Rolling RMS — what the app sees', xaxis_title='Time (ms)', yaxis_title='RMS', height=400, **DARK)
fig.show()

Method:    additive
Chunks:    364  (10ms each)
Baseline:  0.002914  (50th percentile)
Threshold: 0.029144  (10× baseline)
Breakaway: 0.008743  (3× baseline)


## Spike Detection + Lookback

In [7]:
def compute_hf_energy(chunk, bin_low, bin_high):
    N = len(chunk)
    spectrum = np.abs(np.fft.rfft(chunk)) / N
    hi = min(bin_high, len(spectrum) - 1)
    return float(np.sum(spectrum[bin_low:hi+1]))

spikes = []    # (chunk_index, rms, hf_energy, gated)
last_spike_chunk = -999
debounce_chunks  = DEBOUNCE_MS / CHUNK_MS

for i in range(n_chunks):
    rms = rms_vals[i]
    if rms < threshold:
        continue
    chunk_data = combined[i*chunk_samples:(i+1)*chunk_samples]
    hf = compute_hf_energy(chunk_data, HF_BIN_LOW, HF_BIN_HIGH)
    if hf < HF_FLOOR:
        spikes.append({'i': i, 'rms': rms, 'hf': hf, 'gated': True})
        continue
    if i - last_spike_chunk < debounce_chunks:
        continue
    last_spike_chunk = i
    spikes.append({'i': i, 'rms': rms, 'hf': hf, 'gated': False})

live_spikes = [s for s in spikes if not s['gated']]
gated       = [s for s in spikes if s['gated']]
print(f'Spikes above threshold: {len(live_spikes)}  |  Gated by FFT: {len(gated)}')

# ── Lookback pairing ──────────────────────────────────────────────────────────
min_chunks = MIN_LOOKBACK_MS / CHUNK_MS
max_chunks = MAX_LOOKBACK_MS / CHUNK_MS
cycles = []

for idx, tend_spike in enumerate(live_spikes):
    tend_i = tend_spike['i']
    best = None
    for prev in live_spikes[:idx]:
        gap = tend_i - prev['i']
        if gap < min_chunks or gap > max_chunks:
            continue
        if prev['rms'] < bkwy_thr:
            continue
        if best is None or prev['rms'] > best['rms']:
            best = prev
    if best is not None:
        delta_ms = (tend_i - best['i']) * CHUNK_MS
        speed    = (STROKE_IN / delta_ms) * 1000
        cycles.append({
            'tstart_i': best['i'], 'tend_i': tend_i,
            'delta_ms': delta_ms,  'speed': speed,
            'tstart_rms': best['rms'], 'tend_rms': tend_spike['rms']
        })
        print(f"  Cycle: T_start={best['i']*CHUNK_MS:.0f}ms  T_end={tend_i*CHUNK_MS:.0f}ms  "
              f"Δ={delta_ms:.1f}ms  speed={speed:.2f} in/s")
    else:
        print(f"  Unmatched spike at {tend_i*CHUNK_MS:.0f}ms (no T_start in lookback)")

Spikes above threshold: 2  |  Gated by FFT: 1
  Unmatched spike at 910ms (no T_start in lookback)
  Unmatched spike at 2230ms (no T_start in lookback)


## Detection Map — full picture

In [8]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=t_chunks, y=rms_vals, mode='lines', line=dict(color='#4fc3f7', width=1), name='RMS', opacity=0.8))
fig.add_hline(y=threshold, line=dict(color='#ef5350', width=1, dash='dash'), annotation_text='Impact threshold')
fig.add_hline(y=bkwy_thr,  line=dict(color='#ffa726', width=1, dash='dot'),  annotation_text='Breakaway threshold')
fig.add_hline(y=baseline,  line=dict(color='#444444', width=1),              annotation_text='Baseline')

for s in gated:
    fig.add_vline(x=s['i']*CHUNK_MS, line=dict(color='#666666', width=0.8, dash='dot'))

for c in cycles:
    ts_ms = c['tstart_i'] * CHUNK_MS
    te_ms = c['tend_i']   * CHUNK_MS
    fig.add_vline(x=ts_ms, line=dict(color='#ffa726', width=2), annotation_text='T_start')
    fig.add_vline(x=te_ms, line=dict(color='#66bb6a', width=2), annotation_text='T_end')
    fig.add_vrect(x0=ts_ms, x1=te_ms, fillcolor='#66bb6a', opacity=0.08, line_width=0,
                  annotation_text=f"Δ{c['delta_ms']:.0f}ms  {c['speed']:.1f}in/s",
                  annotation_position='top left', annotation_font_color='#ffffff')

fig.update_layout(title='Detection Map', xaxis_title='Time (ms)', yaxis_title='RMS', height=450, **DARK)
fig.show()

print(f'\nSummary: {len(cycles)} cycle(s) detected')
for c in cycles:
    print(f"  Δ={c['delta_ms']:.1f}ms  speed={c['speed']:.3f} in/s  "
          f"T_start RMS={c['tstart_rms']:.5f}  T_end RMS={c['tend_rms']:.5f}")


Summary: 0 cycle(s) detected


## FFT — frequency content of detected events

In [9]:
if not cycles:
    print('No cycles detected — nothing to show FFTs for.')
else:
    for ci, c in enumerate(cycles):
        fig = make_subplots(rows=1, cols=2,
                            subplot_titles=('T_start (breakaway)', 'T_end (impact)'))
        for col, spike_i, color in [(1, c['tstart_i'], '#ffa726'), (2, c['tend_i'], '#66bb6a')]:
            chunk_data = combined[spike_i*chunk_samples:(spike_i+1)*chunk_samples]
            freqs = np.fft.rfftfreq(len(chunk_data), d=1/sr) / 1000
            mags  = np.abs(np.fft.rfft(chunk_data)) / len(chunk_data)
            hf_low_khz  = HF_BIN_LOW  * sr / chunk_samples / 1000
            hf_high_khz = HF_BIN_HIGH * sr / chunk_samples / 1000
            fig.add_trace(go.Scatter(x=freqs, y=mags, mode='lines',
                                     line=dict(color=color, width=0.8), name='Magnitude'), row=1, col=col)
            fig.add_vrect(x0=hf_low_khz, x1=hf_high_khz, fillcolor=color, opacity=0.1,
                          line_width=0, annotation_text='HF gate', row=1, col=col)
        fig.update_layout(title=f'Cycle {ci+1} — FFT', height=380, **DARK)
        fig.update_xaxes(title_text='Frequency (kHz)', gridcolor='#2a2a2a')
        fig.update_yaxes(title_text='Magnitude', gridcolor='#2a2a2a')
        fig.show()

No cycles detected — nothing to show FFTs for.


## Suggested Settings
Based on what was detected — compare these against the app's Settings tab.

In [10]:
if not live_spikes:
    print('No spikes detected — lower IMPACT_MULT or check mounting.')
else:
    rms_values   = [s['rms'] for s in live_spikes]
    hf_values    = [s['hf']  for s in live_spikes]
    rms_min      = min(rms_values)
    rms_min_mult = rms_min / baseline
    hf_min       = min(hf_values)
    MARGIN       = 0.6

    sug_impact    = max(1.5, round(rms_min_mult * MARGIN, 1))
    sug_breakaway = max(1.0, round(sug_impact * 0.5, 1))
    sug_hf        = round(hf_min * 0.8, 4)

    print('── Suggested settings (copy to app Settings tab) ──')
    print(f'  Impact multiplier:    {sug_impact}')
    print(f'  Breakaway multiplier: {sug_breakaway}')
    print(f'  HF Floor:             {sug_hf}')
    print(f'  (Based on {MARGIN*100:.0f}% of worst-case event, baseline={baseline:.6f})')
    if cycles:
        deltas = [c['delta_ms'] for c in cycles]
        print(f'\n── Lookback window recommendation ──')
        print(f'  Measured delta range: {min(deltas):.1f} – {max(deltas):.1f} ms')
        print(f'  Suggested MIN_LOOKBACK_MS: {max(5, int(min(deltas)*0.5))}')
        print(f'  Suggested MAX_LOOKBACK_MS: {int(max(deltas)*1.5)}')

── Suggested settings (copy to app Settings tab) ──
  Impact multiplier:    9.699999809265137
  Breakaway multiplier: 4.800000190734863
  HF Floor:             0.0276
  (Based on 60% of worst-case event, baseline=0.002914)
